In [0]:
-- Game - Skater Stats
-- Full stat line for every skater in one game. Set team for one side only.
-- Change the values in params only; everything below reads from it.
WITH params AS (                              -- change settings here only
  SELECT
    2026020262 AS game_id,          -- e.g. 2025020001 (season 2025, 02 = regular season, game 0001); see Game - Find Game ID
    CAST(NULL AS STRING) AS team    -- team abbreviation, e.g. 'MTL'; NULL = both teams
),
game_players AS (                        -- everyone dressed in the game (game rosters)
  SELECT r.game_id, r.player_id, concat_ws(' ', r.first_name, r.last_name) AS full_name,
         r.position_code, r.sweater_number, r.team_id, t.abbrev AS team,
         r.team_id = g.home_team_id AS is_home, g.home_team_id
  FROM nhl.silver.game_rosters r
  JOIN nhl.gold.dim_game g ON r.game_id = g.game_id
  JOIN nhl.gold.dim_team t ON r.team_id = t.team_id
  CROSS JOIN params pr
  WHERE r.game_id = pr.game_id
),
skaters AS (
  SELECT gp.*
  FROM game_players gp
  CROSS JOIN params pr
  WHERE gp.position_code <> 'G'
    AND (pr.team IS NULL OR gp.team = pr.team)
),
events AS (                              -- plays where one of those players had a role
  SELECT sk.player_id AS pid, f.*
  FROM nhl.gold.fact_event f
  JOIN skaters sk
    ON f.game_id = sk.game_id
   AND sk.player_id IN (f.scoring_player_id, f.assist1_player_id, f.assist2_player_id,
                        f.shooting_player_id, f.blocking_player_id, f.hitting_player_id,
                        f.hittee_player_id, f.winning_player_id, f.losing_player_id,
                        f.committed_by_player_id, f.drawn_by_player_id, f.player_id)
  WHERE COALESCE(f.period_type, '') <> 'SO'    -- shootout attempts aren't stats
),
stats AS (
  SELECT sk.team AS team,
         sk.is_home AS is_home,
         sk.sweater_number AS sweater_number,
         sk.full_name AS full_name,
         sk.position_code AS position,
         COUNT_IF(e.type_desc_key = 'goal' AND e.scoring_player_id = e.pid) AS goals,
         COUNT_IF(e.type_desc_key = 'goal' AND e.pid IN (e.assist1_player_id, e.assist2_player_id)) AS assists,
         COUNT_IF((e.type_desc_key = 'goal' AND e.scoring_player_id = e.pid)
               OR (e.type_desc_key = 'shot-on-goal' AND e.shooting_player_id = e.pid)) AS shots,
         COUNT_IF(e.type_desc_key = 'missed-shot' AND e.shooting_player_id = e.pid) AS missed_shots,
         COUNT_IF(e.type_desc_key = 'blocked-shot' AND e.shooting_player_id = e.pid) AS own_shots_blocked,
         COUNT_IF(e.type_desc_key = 'hit' AND e.hitting_player_id = e.pid) AS hits,
         COUNT_IF(e.type_desc_key = 'blocked-shot' AND e.blocking_player_id = e.pid) AS blocked_shots,
         COUNT_IF(e.type_desc_key = 'takeaway' AND e.player_id = e.pid) AS takeaways,
         COUNT_IF(e.type_desc_key = 'giveaway' AND e.player_id = e.pid) AS giveaways,
         SUM(IF(e.type_desc_key = 'penalty' AND e.committed_by_player_id = e.pid,
               COALESCE(e.penalty_duration, 0), 0)) AS pim,
         COUNT_IF(e.type_desc_key = 'faceoff' AND e.winning_player_id = e.pid) AS faceoff_wins,
         COUNT_IF(e.type_desc_key = 'faceoff' AND e.losing_player_id = e.pid) AS faceoff_losses
  FROM skaters sk
  LEFT JOIN events e ON e.pid = sk.player_id
  GROUP BY sk.team, sk.is_home, sk.sweater_number, sk.full_name, sk.position_code, sk.player_id
)
SELECT team,
       sweater_number,
       full_name,
       position,
       goals,
       assists,
       goals + assists AS points,
       shots,
       (shots + missed_shots + own_shots_blocked) AS shot_attempts,
       hits,
       blocked_shots,
       takeaways,
       giveaways,
       pim,
       faceoff_wins,
       faceoff_losses,
       ROUND(100.0 * faceoff_wins / NULLIF(faceoff_wins + faceoff_losses, 0), 1) AS faceoff_pct
FROM stats
ORDER BY is_home, points DESC, goals DESC, shots DESC;